In [ ]:
import pandas as pd
import numpy as np
import mlflow
import matplotlib.pyplot as plt
from mlflow.models import infer_signature
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, ConfusionMatrixDisplay, PrecisionRecallDisplay

# Carregando os dados pré-processados
X_train = pd.read_csv('dados/X_train.csv')
X_test = pd.read_csv('dados/X_test.csv')
y_train = pd.read_csv('dados/y_train.csv').values.ravel() # .ravel() para transformar em array 1D
y_test = pd.read_csv('dados/y_test.csv').values.ravel()

# Configurando o experimento no MLflow
mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("Projeto_Agua_Potavel")

In [ ]:
def treinar_e_avaliar_modelo(nome_modelo, modelo_base, param_grid, X_train, y_train, X_test, y_test):
    print(f"--- Treinando {nome_modelo} ---")

    # Inicia uma run no MLflow
    with mlflow.start_run(run_name=nome_modelo):

        # Otimização de Hiperparâmetros com validação cruzada (CV=5)
        # Usamos 'recall' como métrica principal de otimização, pois é crucial para saúde
        grid_search = GridSearchCV(estimator=modelo_base, param_grid=param_grid,
                                   cv=5, scoring='recall', n_jobs=-1, verbose=1)
        grid_search.fit(X_train, y_train)

        # Melhor modelo encontrado
        melhor_modelo = grid_search.best_estimator_
        print(f"Melhores parâmetros: {grid_search.best_params_}")

        # Previsões nos dados de teste
        y_pred = melhor_modelo.predict(X_test)

        # Calculando métricas
        acc = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred)
        rec = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)

        print("\nMétricas no Conjunto de Teste:")
        print(f"Acurácia: {acc:.4f}")
        print(f"Precisão: {prec:.4f}")
        print(f"Recall: {rec:.4f}")
        print(f"F1-Score: {f1:.4f}\n")
        print(classification_report(y_test, y_pred))

        # Registrando parâmetros e métricas no MLflow
        mlflow.log_params(grid_search.best_params_)
        mlflow.log_param("cv_folds", 5)
        mlflow.log_param("scoring", "recall")
        mlflow.log_metric("cv_best_recall", grid_search.best_score_)
        mlflow.log_metric("recall", rec)
        mlflow.log_metric("f1_score", f1)
        mlflow.log_metric("accuracy", acc)
        mlflow.log_metric("precision", prec)
        mlflow.set_tag("autor", "equipe")
        mlflow.set_tag("dataset", "water_potability_limpo + SMOTE")

        # Salvando o modelo no MLflow e registrando no Model Registry (versionamento).
        # skops_trusted_types: o MLflow serializa com skops (auditoria de segurança), que
        # rejeita tipos não-nativos do sklearn. O XGBoost traz tipos próprios, então
        # precisam ser declarados como confiáveis (ignorado para LR/RF, que não os usam).
        signature = infer_signature(X_test, y_pred)
        mlflow.sklearn.log_model(
            sk_model=melhor_modelo,
            name=f"modelo_{nome_modelo.replace(' ', '_')}",
            signature=signature,
            input_example=X_test.iloc[:5],
            registered_model_name=f"potabilidade_{nome_modelo.replace(' ', '_')}",
            skops_trusted_types=["xgboost.sklearn.XGBClassifier", "xgboost.core.Booster"],
        )

        # Matriz de confusão
        fig_cm, ax_cm = plt.subplots()
        ConfusionMatrixDisplay.from_predictions(y_test, y_pred, ax=ax_cm)
        ax_cm.set_title(f"Matriz de Confusão — {nome_modelo}")
        mlflow.log_figure(fig_cm, "confusion_matrix.png")
        plt.close(fig_cm)

        # Curva Precision-Recall (relevante pois otimizamos recall)
        fig_pr, ax_pr = plt.subplots()
        PrecisionRecallDisplay.from_estimator(melhor_modelo, X_test, y_test, ax=ax_pr)
        ax_pr.set_title(f"Curva Precision-Recall — {nome_modelo}")
        mlflow.log_figure(fig_pr, "precision_recall_curve.png")
        plt.close(fig_pr)

        # Relatório completo como texto
        mlflow.log_text(classification_report(y_test, y_pred), "classification_report.txt")

        return melhor_modelo

In [ ]:
param_grid_lr = {
    'C': [0.01, 0.1, 1, 10, 100],
    'solver': ['liblinear', 'lbfgs'],
    'max_iter': [500, 1000] # Aumentado para garantir convergência
}

lr_model = LogisticRegression(class_weight='balanced', random_state=42)

melhor_lr = treinar_e_avaliar_modelo("Regressao Logistica", lr_model, param_grid_lr, X_train, y_train, X_test, y_test)

In [ ]:
param_grid_rf = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

rf_model = RandomForestClassifier(class_weight='balanced', random_state=42)

melhor_rf = treinar_e_avaliar_modelo("Random Forest", rf_model, param_grid_rf, X_train, y_train, X_test, y_test)

In [ ]:
param_grid_xgb = {
    'n_estimators': [100, 200],
    'learning_rate': [0.01, 0.1, 0.2],
    'max_depth': [3, 5, 7],
    'subsample': [0.8, 1.0]
}

# scale_pos_weight ajuda a lidar com o desbalanceamento das classes
# Deve ser aprox: total_negativos / total_positivos
xgb_model = XGBClassifier(eval_metric='logloss', random_state=42)

melhor_xgb = treinar_e_avaliar_modelo("XGBoost", xgb_model, param_grid_xgb, X_train, y_train, X_test, y_test)